# Machine Learning Zoomcamp 2026 - Homework 6
## Decision Trees and Ensemble Learning

In this homework, we build regression models to predict car fuel efficiency (`fuel_efficiency_mpg`) using:
- Decision Tree Regressor
- Random Forest Regressor
- XGBoost Regressor

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import xgboost as xgb

# 1. Load dataset
df = pd.read_csv('car_fuel_efficiency_2026_homework6.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Check missing values
print('Missing values before imputation:')
print(df.isnull().sum())

# Fill missing values with zeros
df = df.fillna(0)

In [ ]:
# Split dataset
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

print(f'Train: {len(df_train)}, Validation: {len(df_val)}, Test: {len(df_test)}')

y_train = df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values
y_test = df_test['fuel_efficiency_mpg'].values

del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

# DictVectorizer
dv = DictVectorizer(sparse=True)
train_dicts = df_train.to_dict(orient='records')
val_dicts = df_val.to_dict(orient='records')
test_dicts = df_test.to_dict(orient='records')

X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)
X_test = dv.transform(test_dicts)

feature_names = dv.get_feature_names_out()
print(f'Number of features: {len(feature_names)}')
print(feature_names)

## Question 1: Decision Tree Regressor (max_depth=1)
Which original feature is used for splitting the data?

In [ ]:
dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)

split_feature_idx = dt.tree_.feature[0]
split_feature_name = feature_names[split_feature_idx]
original_feature = split_feature_name.split('=')[0] if '=' in split_feature_name else split_feature_name

print(f'Splitting feature index: {split_feature_idx}')
print(f'Splitting feature name: {split_feature_name}')
print(f'Original feature name: {original_feature}')

## Question 2: Random Forest Regressor (n_estimators=10)
What is the RMSE of this model on the validation data?

In [ ]:
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred_val = rf.predict(X_val)
rmse_q2 = np.sqrt(mean_squared_error(y_val, y_pred_val))
print(f'Validation RMSE: {rmse_q2:.4f}')

## Question 3: Tuning n_estimators
Experiment with `n_estimators` in `[10, 50, 100, 150]` with `random_state=1`.

In [ ]:
n_estimators_list = [10, 50, 100, 150]

for n in n_estimators_list:
    rf_n = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    rf_n.fit(X_train, y_train)
    y_pred = rf_n.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    print(f'n_estimators = {n:>3d} -> Validation RMSE = {rmse:.4f}')

## Question 4: Tuning max_depth
Find the best `max_depth` from `[10, 15, 20, 25]` using the mean RMSE across `n_estimators` in `[10, 50, 100, 150]`.

In [ ]:
max_depth_list = [10, 15, 20, 25]

for depth in max_depth_list:
    rmse_scores = []
    for n in n_estimators_list:
        rf_d = RandomForestRegressor(n_estimators=n, max_depth=depth, random_state=1, n_jobs=-1)
        rf_d.fit(X_train, y_train)
        y_pred = rf_d.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        rmse_scores.append(rmse)
    print(f'max_depth = {depth:>2d} -> Mean RMSE = {np.mean(rmse_scores):.4f} (scores: {[round(s, 4) for s in rmse_scores]})')

## Question 5: Feature Importance
Find the most important feature (among `vehicle_weight`, `horsepower`, `acceleration`, `engine_displacement`).

In [ ]:
rf_imp = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)
rf_imp.fit(X_train, y_train)

importances = pd.DataFrame({
    'feature': feature_names,
    'importance': rf_imp.feature_importances_
}).sort_values('importance', ascending=False)

print('Top 10 Feature importances:')
print(importances.head(10))

candidates = ['vehicle_weight', 'horsepower', 'acceleration', 'engine_displacement']
print('\nCandidate features:')
print(importances[importances['feature'].isin(candidates)])

## Question 6: XGBoost (eta=0.3 vs eta=0.1)
Train XGBoost for 100 rounds with `eta=0.3` and `eta=0.1`.

In [ ]:
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=list(feature_names))
dval = xgb.DMatrix(X_val, label=y_val, feature_names=list(feature_names))

watchlist = [(dtrain, 'train'), (dval, 'val')]

# eta = 0.3
xgb_params_03 = {
    'eta': 0.3, 
    'max_depth': 6,
    'min_child_weight': 1,
    'objective': 'reg:squarederror',
    'nthread': 8,
    'seed': 1,
    'verbosity': 1,
}

evals_03 = {}
model_03 = xgb.train(xgb_params_03, dtrain, num_boost_round=100, evals=watchlist, evals_result=evals_03, verbose_eval=False)

# eta = 0.1
xgb_params_01 = {
    'eta': 0.1, 
    'max_depth': 6,
    'min_child_weight': 1,
    'objective': 'reg:squarederror',
    'nthread': 8,
    'seed': 1,
    'verbosity': 1,
}

evals_01 = {}
model_01 = xgb.train(xgb_params_01, dtrain, num_boost_round=100, evals=watchlist, evals_result=evals_01, verbose_eval=False)

print(f'eta=0.3 -> Final validation RMSE: {evals_03["val"]["rmse"][-1]:.4f}')
print(f'eta=0.1 -> Final validation RMSE: {evals_01["val"]["rmse"][-1]:.4f}')